In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import f1_score
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.10.0


In [4]:
import os
print(os.listdir('../data/'))

['.DS_Store', 'stable_probes_783.csv', '.gitkeep', 'stable_probes_758.csv', 'processed', 'raw', 'stable_probes_100.csv']


In [5]:
print(os.listdir('../data/processed/'))

['ml_ready.csv', 'ml_ready_max_samples.csv', 'patient_groups_max_samples.csv', '.ipynb_checkpoints']


In [12]:
probes_100 = pd.read_csv('../data/stable_probes_100.csv')['0'].tolist()
print(f"Loaded {len(probes_100)} probes")

Loaded 100 probes


In [7]:
# Load data
df = pd.read_csv('../data/processed/ml_ready_max_samples.csv', index_col=0)
patient_groups = pd.read_csv('../data/processed/patient_groups_max_samples.csv', index_col=0)
stable_probes = pd.read_csv('../data/stable_probes_100.csv')['0'].tolist()

groups = patient_groups['patient_group']

le = LabelEncoder()
y_encoded = le.fit_transform(df['diagnosis'])

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Samples: {df.shape[0]}")
print(f"Stable probes: {len(stable_probes)}")
print(f"Classes: {dict(zip(le.classes_, np.bincount(y_encoded)))}")

Samples: 607
Stable probes: 100
Classes: {'BD': 134, 'CTL': 227, 'MDD': 75, 'SCZ': 171}


In [8]:
# --- WGAN-GP for Gene Expression Data ---
# Wasserstein GAN with Gradient Penalty
# More stable than basic GAN on small datasets
# Critic (discriminator) outputs a score, not a probability

n_probes = len(stable_probes)
NOISE_DIM = 64

# Generator: random noise → fake gene expression
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(NOISE_DIM, 128),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, n_probes)
        )
    
    def forward(self, z):
        return self.model(z)

# Critic: gene expression → score (higher = more real)
# No Sigmoid at the end (unlike basic GAN)
class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(n_probes, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 1)       # No Sigmoid: outputs raw score
        )
    
    def forward(self, x):
        return self.model(x)

# Gradient Penalty: keeps critic smooth and stable
def gradient_penalty(critic, real, fake):
    batch_size = real.shape[0]
    epsilon = torch.rand(batch_size, 1)
    interpolated = epsilon * real + (1 - epsilon) * fake
    interpolated.requires_grad_(True)
    
    scores = critic(interpolated)
    gradients = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True,
        retain_graph=True
    )[0]
    
    gradient_norm = gradients.norm(2, dim=1)
    penalty = ((gradient_norm - 1) ** 2).mean()
    return penalty

print(f"WGAN-GP ready: Generator noise({NOISE_DIM}) → expression({n_probes})")
print(f"Critic: expression({n_probes}) → real/fake score")

WGAN-GP ready: Generator noise(64) → expression(100)
Critic: expression(100) → real/fake score


In [9]:
# --- Train WGAN-GP per class ---

def train_wgan_gp(real_data, n_epochs=1000, n_generate=200, 
                   lr=0.0001, n_critic=5, lambda_gp=10):
    """
    Train WGAN-GP on real gene expression data and generate synthetic samples.
    
    real_data: numpy array (n_samples, n_probes)
    n_epochs: training iterations
    n_generate: how many synthetic samples to produce
    n_critic: train critic this many times per generator update
    lambda_gp: gradient penalty weight
    """
    
    # Scale data for stable training
    scaler = StandardScaler()
    real_scaled = scaler.fit_transform(real_data)
    real_tensor = torch.FloatTensor(real_scaled)
    
    # Initialize models
    gen = Generator()
    critic = Critic()
    
    # Optimizers (Adam with WGAN-GP recommended betas)
    opt_gen = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_critic = torch.optim.Adam(critic.parameters(), lr=lr, betas=(0.0, 0.9))
    
    # Training loop
    for epoch in range(n_epochs):
        
        # --- Train Critic n_critic times per generator update ---
        for _ in range(n_critic):
            noise = torch.randn(real_tensor.shape[0], NOISE_DIM)
            fake = gen(noise).detach()
            
            critic_real = critic(real_tensor).mean()
            critic_fake = critic(fake).mean()
            gp = gradient_penalty(critic, real_tensor, fake)
            
            # Wasserstein loss + gradient penalty
            loss_critic = critic_fake - critic_real + lambda_gp * gp
            
            opt_critic.zero_grad()
            loss_critic.backward()
            opt_critic.step()
        
        # --- Train Generator ---
        noise = torch.randn(real_tensor.shape[0], NOISE_DIM)
        fake = gen(noise)
        loss_gen = -critic(fake).mean()
        
        opt_gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()
        
        if (epoch + 1) % 200 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs} - Critic: {loss_critic.item():.4f}, Gen: {loss_gen.item():.4f}")
    
    # Generate synthetic samples
    gen.eval()
    with torch.no_grad():
        noise = torch.randn(n_generate, NOISE_DIM)
        synthetic_scaled = gen(noise).numpy()
    
    synthetic = scaler.inverse_transform(synthetic_scaled)
    return synthetic

print("WGAN-GP training function ready")

WGAN-GP training function ready


In [10]:
# --- XGBoost with WGAN-GP Augmentation inside CV ---
# GAN trains on training data only per fold (no leakage)
# Synthetic samples added to training set only
# Test on real data only

X_real = df[stable_probes].values
target_per_class = 300

fold_f1_train = []
fold_f1_test = []

for fold_num, (train_idx, test_idx) in enumerate(cv.split(df[stable_probes], y_encoded, groups)):
    
    print(f"\n--- Fold {fold_num + 1} ---")
    
    X_train = X_real[train_idx]
    y_train = y_encoded[train_idx]
    X_test = X_real[test_idx]
    y_test = y_encoded[test_idx]
    
    # Generate synthetic samples for each class from training data only
    aug_X = [X_train]
    aug_y = [y_train]
    
    for class_idx, class_name in enumerate(le.classes_):
        mask = y_train == class_idx
        real_class = X_train[mask]
        n_real = real_class.shape[0]
        n_generate = target_per_class - n_real
        
        if n_generate <= 0:
            continue
        
        print(f"  {class_name}: {n_real} real → generating {n_generate} synthetic")
        synthetic = train_wgan_gp(real_class, n_epochs=500, n_generate=n_generate)
        aug_X.append(synthetic)
        aug_y.append(np.full(n_generate, class_idx))
    
    # Combine real + synthetic training data
    X_train_aug = np.vstack(aug_X)
    y_train_aug = np.concatenate(aug_y)
    print(f"  Augmented training: {X_train_aug.shape[0]} samples (was {X_train.shape[0]})")
    
    # Train XGBoost on augmented data
    model = xgb.XGBClassifier(
        n_estimators=25, max_depth=2, learning_rate=0.3,
        eval_metric='mlogloss', random_state=42
    )
    model.fit(X_train_aug, y_train_aug)
    
    # Evaluate on real data only
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    fold_f1_train.append(train_f1)
    fold_f1_test.append(test_f1)
    
    print(f"  Train F1: {train_f1:.3f}, Test F1: {test_f1:.3f}")

print(f"\n{'='*50}")
print(f"WGAN-GP + XGBoost (25 trees, depth=2)")
print(f"Train: {np.mean(fold_f1_train):.3f}, Test: {np.mean(fold_f1_test):.3f}")


--- Fold 1 ---
  BD: 107 real → generating 193 synthetic
  Epoch 200/500 - Critic: -1.9998, Gen: -2.4299
  Epoch 400/500 - Critic: -1.8073, Gen: -0.8575
  CTL: 181 real → generating 119 synthetic
  Epoch 200/500 - Critic: -2.0866, Gen: -1.6647
  Epoch 400/500 - Critic: -1.7960, Gen: -0.5002
  MDD: 60 real → generating 240 synthetic
  Epoch 200/500 - Critic: -1.8127, Gen: -0.0791
  Epoch 400/500 - Critic: -2.7725, Gen: 1.2571
  SCZ: 137 real → generating 163 synthetic
  Epoch 200/500 - Critic: -2.0981, Gen: -1.7996
  Epoch 400/500 - Critic: -1.7972, Gen: -0.2996
  Augmented training: 1200 samples (was 485)
  Train F1: 0.847, Test F1: 0.536

--- Fold 2 ---
  BD: 107 real → generating 193 synthetic
  Epoch 200/500 - Critic: -2.0442, Gen: -2.1275
  Epoch 400/500 - Critic: -1.9061, Gen: -0.7846
  CTL: 182 real → generating 118 synthetic
  Epoch 200/500 - Critic: -2.1654, Gen: -1.8009
  Epoch 400/500 - Critic: -1.8330, Gen: -0.6395
  MDD: 60 real → generating 240 synthetic
  Epoch 200/500 -

In [15]:
from sklearn.linear_model import LogisticRegression

In [16]:
# --- WGAN-GP + LR on 100 stable probes ---
# --- WGAN-GP + LR on 100 stable probes ---

probes_100 = pd.read_csv('../data/stable_probes_100.csv')['0'].tolist()
print(f"Loaded {len(probes_100)} probes")

fold_f1_train = []
fold_f1_test = []
target_per_class = 300

for fold_num, (train_idx, test_idx) in enumerate(cv.split(df[probes_100], y_encoded, groups)):
    
    print(f"\n--- Fold {fold_num + 1} ---")
    
    X_train = df[probes_100].iloc[train_idx].values
    y_train = y_encoded[train_idx]
    X_test = df[probes_100].iloc[test_idx].values
    y_test = y_encoded[test_idx]
    
    aug_X = [X_train]
    aug_y = [y_train]
    
    for class_idx, class_name in enumerate(le.classes_):
        mask = y_train == class_idx
        real_class = X_train[mask]
        n_real = real_class.shape[0]
        n_generate = target_per_class - n_real
        
        if n_generate <= 0:
            continue
        
        print(f"  {class_name}: {n_real} real → generating {n_generate} synthetic")
        synthetic = train_wgan_gp(real_class, n_epochs=500, n_generate=n_generate)
        aug_X.append(synthetic)
        aug_y.append(np.full(n_generate, class_idx))
    
    X_train_aug = np.vstack(aug_X)
    y_train_aug = np.concatenate(aug_y)
    print(f"  Augmented training: {X_train_aug.shape[0]} samples")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_aug)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression(class_weight='balanced', max_iter=5000)
    model.fit(X_train_scaled, y_train_aug)
    
    X_train_real_scaled = scaler.transform(X_train)
    y_pred_train = model.predict(X_train_real_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    fold_f1_train.append(train_f1)
    fold_f1_test.append(test_f1)
    
    print(f"  Train F1: {train_f1:.3f}, Test F1: {test_f1:.3f}")

print(f"\n{'='*50}")
print(f"WGAN-GP + LR (100 stable probes)")
print(f"Train: {np.mean(fold_f1_train):.3f}, Test: {np.mean(fold_f1_test):.3f}")

Loaded 100 probes

--- Fold 1 ---
  BD: 107 real → generating 193 synthetic
  Epoch 200/500 - Critic: -1.8937, Gen: -2.0427
  Epoch 400/500 - Critic: -1.8834, Gen: -0.0494
  CTL: 181 real → generating 119 synthetic
  Epoch 200/500 - Critic: -1.8574, Gen: -2.1372
  Epoch 400/500 - Critic: -1.6616, Gen: -1.2002
  MDD: 60 real → generating 240 synthetic
  Epoch 200/500 - Critic: -2.5922, Gen: 0.0356
  Epoch 400/500 - Critic: -2.6200, Gen: 0.6915
  SCZ: 137 real → generating 163 synthetic
  Epoch 200/500 - Critic: -2.3767, Gen: -1.6076
  Epoch 400/500 - Critic: -1.9576, Gen: -0.2190
  Augmented training: 1200 samples
  Train F1: 0.755, Test F1: 0.527

--- Fold 2 ---
  BD: 107 real → generating 193 synthetic
  Epoch 200/500 - Critic: -2.0190, Gen: -1.4905
  Epoch 400/500 - Critic: -2.2297, Gen: -0.0885
  CTL: 182 real → generating 118 synthetic
  Epoch 200/500 - Critic: -1.9267, Gen: -2.0636
  Epoch 400/500 - Critic: -1.6003, Gen: -0.5805
  MDD: 60 real → generating 240 synthetic
  Epoch 20